# Example 6 — Bike Demand Predictor
**Predict the number of bike rentals during a future hour**  
**Difficulty:** Intermediate

Imagine that you help operate a bicycle-sharing service. Before each hour begins, the team would like an estimate of how many rental transactions may occur.

You have historical hourly data describing:

- time of day,
- day of the week,
- season,
- holiday status,
- temperature,
- humidity,
- wind,
- rain,
- snow,
- actual bike rentals.

Your goal is to build a regression model that predicts hourly bike-rental demand.

### Your mission

1. Explore the historical bike-demand data.
2. identify useful model inputs.
3. split complete dates into training, validation, and test groups.
4. compare regression models against a simple baseline.
5. select a model using validation data.
6. evaluate the selected model on unseen test dates.
7. inspect prediction errors and model explanations.
8. enter one future weather-and-time scenario.
9. generate a numerical demand prediction.

The prediction target is the number of **rental transactions during one hour**.

## Colab use

Run the cells from top to bottom. Free Google Colab CPU is sufficient.

The instructor should configure the dataset location once in the data-loading cell. Students should not need to edit that setup.

No external service is required.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

print('Libraries imported successfully.')

## Load the dataset

The prepared dataset is stored as a CSV file. Each row represents one hour of bike-sharing activity.

The instructor can use either:

- a direct public link to the CSV file, or
- a shared file in Google Drive.

Only the configuration values at the top of the next cell should need editing.

In [ ]:
DATA_URL = ''

# Edit the shared Drive path.
DRIVE_DATA_FILE = (
    '/content/drive/MyDrive/AI Summer Camp/data/'
    'seoul_bike_demand.csv'
)


if DATA_URL.strip():
    DATA_FILE = Path(
        '/content/seoul_bike_demand.csv'
    )

    urlretrieve(
        DATA_URL,
        DATA_FILE,
    )

    print('Using direct dataset download.')

else:
    try:
        from google.colab import drive
    except ModuleNotFoundError as error:
        raise RuntimeError(
            'This Drive-loading option must run in Google Colab. '
            'Alternatively, set DATA_URL to a direct-download link.'
        ) from error

    drive.mount('/content/drive')
    DATA_FILE = Path(DRIVE_DATA_FILE)

    print('Using the dataset stored in Google Drive.')


print('Dataset location:')
print(DATA_FILE)

In [ ]:
REQUIRED_COLUMNS = [
    'datetime',
    'date',
    'hour',
    'day_of_week',
    'month',
    'season',
    'is_holiday',
    'temperature_c',
    'humidity_percent',
    'wind_speed_m_s',
    'rainfall_mm',
    'snowfall_cm',
    'rented_bike_count',
]


def load_bike_data(csv_path):
    """Load, validate, and sort the prepared bike-demand data."""
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(
            'The dataset was not found.\n'
            f'Expected location: {csv_path}\n\n'
            'Ask the instructor for the shared Drive path or direct URL.'
        )

    loaded_data = pd.read_csv(
        csv_path,
        parse_dates=[
            'datetime',
            'date',
        ],
    )

    missing_columns = sorted(
        set(REQUIRED_COLUMNS)
        - set(loaded_data.columns)
    )

    if missing_columns:
        raise ValueError(
            'The CSV is missing required columns: '
            + ', '.join(missing_columns)
        )

    loaded_data = (
        loaded_data
        .sort_values('datetime')
        .reset_index(drop=True)
    )

    return loaded_data


data = load_bike_data(
    DATA_FILE
)

print('Dataset loaded successfully.')
print(f'Rows: {data.shape[0]:,}')
print(f'Columns: {data.shape[1]}')

print(
    'Date range:',
    data['date'].min().date(),
    'through',
    data['date'].max().date(),
)

In [ ]:
display(
    data.head(10)
)

## Data dictionary

| Column | Meaning |
|---|---|
| `datetime` | Beginning of the hourly observation |
| `date` | Calendar date |
| `hour` | Hour of day from 0 through 23 |
| `time_period` | Readable part-of-day category |
| `day_of_week` | Weekday name |
| `month` | Month number from 1 through 12 |
| `is_weekend` | 1 for Saturday or Sunday |
| `season` | Winter, Spring, Summer, or Autumn |
| `is_holiday` | 1 for a holiday |
| `weather_condition` | Dry, Rain, Snow, or Rain and snow |
| `temperature_c` | Temperature in degrees Celsius |
| `humidity_percent` | Relative humidity percentage |
| `wind_speed_m_s` | Wind speed in meters per second |
| `visibility_m` | Visibility in meters |
| `solar_radiation_mj_m2` | Solar radiation |
| `rainfall_mm` | Rainfall in millimeters |
| `snowfall_cm` | Snowfall in centimeters |
| `rented_bike_count` | Number of rental transactions during the hour |

The target or label is:

`rented_bike_count`

In [ ]:
print('COLUMN TYPES')
print('=' * 70)

display(
    data.dtypes.to_frame(
        'data_type'
    )
)

print('\nMISSING VALUES')
print('=' * 70)

display(
    data
    .isna()
    .sum()
    .to_frame(
        'missing_values'
    )
)

print('\nDUPLICATE DATETIMES')
print('=' * 70)

print(
    data['datetime']
    .duplicated()
    .sum()
)

In [ ]:
numeric_columns = [
    'hour',
    'temperature_c',
    'humidity_percent',
    'wind_speed_m_s',
    'visibility_m',
    'solar_radiation_mj_m2',
    'rainfall_mm',
    'snowfall_cm',
    'rented_bike_count',
]

display(
    data[numeric_columns]
    .describe()
    .round(2)
)

In [ ]:
average_demand = (
    data['rented_bike_count']
    .mean()
)

minimum_demand = (
    data['rented_bike_count']
    .min()
)

maximum_demand = (
    data['rented_bike_count']
    .max()
)

busiest_hour = data.loc[
    data['rented_bike_count']
    .idxmax()
]

print(
    f'Average hourly rentals: '
    f'{average_demand:.2f}'
)

print(
    f'Minimum hourly rentals: '
    f'{minimum_demand:,}'
)

print(
    f'Maximum hourly rentals: '
    f'{maximum_demand:,}'
)

print('\nBusiest observed hour:')

display(
    busiest_hour.to_frame(
        'value'
    )
)

# Exploratory Data Analysis

Before training a model, investigate the data.

Look for:

- changes in demand across the year,
- hours with especially high or low demand,
- differences among weekdays,
- interactions between weekday and hour,
- seasonal patterns,
- relationships between weather and rentals,
- extreme or unusual observations.

Exploration can suggest useful features, but a visible relationship does not automatically prove causation.

In [ ]:
daily_demand = (
    data
    .groupby('date')[
        'rented_bike_count'
    ]
    .sum()
)

plt.figure(
    figsize=(14, 5)
)

plt.plot(
    daily_demand.index,
    daily_demand.values,
    linewidth=1,
)

plt.xlabel('Date')
plt.ylabel('Total rentals during the day')
plt.title('Daily Bike-Rental Demand Over Time')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.hist(
    data['rented_bike_count'],
    bins=30,
    edgecolor='black',
)

plt.xlabel(
    'Rental transactions during one hour'
)

plt.ylabel(
    'Number of hourly observations'
)

plt.title(
    'Distribution of Hourly Bike Demand'
)

plt.tight_layout()
plt.show()

In [ ]:
hourly_summary = (
    data
    .groupby('hour')[
        'rented_bike_count'
    ]
    .agg(
        [
            'count',
            'mean',
            'median',
            'std',
        ]
    )
)

display(
    hourly_summary.round(2)
)

plt.figure(
    figsize=(11, 5)
)

plt.plot(
    hourly_summary.index,
    hourly_summary['mean'],
    marker='o',
)

plt.xlabel('Hour of day')
plt.ylabel('Average hourly rentals')
plt.title('Average Rental Demand by Hour')

plt.xticks(
    range(0, 24)
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
WEEKDAY_ORDER = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday',
]

weekday_summary = (
    data
    .groupby('day_of_week')[
        'rented_bike_count'
    ]
    .agg(
        [
            'count',
            'mean',
            'median',
            'std',
        ]
    )
    .reindex(
        WEEKDAY_ORDER
    )
)

display(
    weekday_summary.round(2)
)

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    weekday_summary.index,
    weekday_summary['mean'],
)

plt.xlabel('Day of week')
plt.ylabel('Average hourly rentals')
plt.title('Average Rental Demand by Day of Week')

plt.xticks(
    rotation=30
)

plt.tight_layout()
plt.show()

In [ ]:
weekday_hour_table = (
    data
    .pivot_table(
        index='day_of_week',
        columns='hour',
        values='rented_bike_count',
        aggfunc='mean',
    )
    .reindex(
        WEEKDAY_ORDER
    )
)

plt.figure(
    figsize=(14, 6)
)

plt.imshow(
    weekday_hour_table,
    aspect='auto',
)

plt.colorbar(
    label='Average hourly rentals'
)

plt.xlabel('Hour of day')
plt.ylabel('Day of week')
plt.title('Average Demand by Weekday and Hour')

plt.xticks(
    ticks=range(24),
    labels=range(24),
)

plt.yticks(
    ticks=range(
        len(WEEKDAY_ORDER)
    ),
    labels=WEEKDAY_ORDER,
)

plt.tight_layout()
plt.show()

In [ ]:
SEASON_ORDER = [
    'Winter',
    'Spring',
    'Summer',
    'Autumn',
]

season_summary = (
    data
    .groupby('season')[
        'rented_bike_count'
    ]
    .agg(
        [
            'count',
            'mean',
            'median',
            'std',
        ]
    )
    .reindex(
        SEASON_ORDER
    )
)

display(
    season_summary.round(2)
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    season_summary.index,
    season_summary['mean'],
)

plt.xlabel('Season')
plt.ylabel('Average hourly rentals')
plt.title('Average Rental Demand by Season')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(9, 6)
)

plt.scatter(
    data['temperature_c'],
    data['rented_bike_count'],
    alpha=0.25,
)

plt.xlabel('Temperature (°C)')
plt.ylabel('Hourly rental transactions')
plt.title('Temperature and Bike Demand')

plt.tight_layout()
plt.show()

In [ ]:
correlation_columns = [
    'hour',
    'is_weekend',
    'is_holiday',
    'temperature_c',
    'humidity_percent',
    'wind_speed_m_s',
    'visibility_m',
    'solar_radiation_mj_m2',
    'rainfall_mm',
    'snowfall_cm',
    'rented_bike_count',
]

demand_correlations = (
    data[
        correlation_columns
    ]
    .corr()[
        'rented_bike_count'
    ]
    .sort_values(
        ascending=False
    )
)

display(
    demand_correlations
    .round(3)
    .to_frame(
        'correlation_with_demand'
    )
)

## EDA discussion

Discuss these questions with your team:

1. At approximately what times does demand peak?
2. Are the morning and evening patterns similar?
3. Which season has the highest average demand?
4. How does rain appear to relate to demand?
5. Does temperature have a perfectly straight-line relationship with demand?
6. Why might the same hour behave differently on different weekdays?
7. Which inputs might be especially useful to a model?
8. Which important variables might be missing?

Possible missing information includes nearby events, transit disruptions, bike availability, road construction, prices, and promotions.

# Regression in Friendly Language

Regression predicts a numerical value. Here, the target is hourly bike rentals.

A linear regression model has the general form

$
\hat{y}
=
b
+
w_1x_1
+
w_2x_2
+\cdots+
w_px_p.
$

A random forest combines predictions from many decision trees and can learn nonlinear patterns.

### Mean absolute error

$
\operatorname{MAE}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left|y_i-\hat{y}_i\right|.
$

MAE is the average absolute mistake, measured in rental transactions per hour.

### Root mean squared error

$
\operatorname{RMSE}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat{y}_i)^2
}.
$

RMSE penalizes large mistakes more strongly than MAE.

### R-squared

$
R^2
=
1-
\frac{
\sum_i(y_i-\hat{y}_i)^2
}{
\sum_i(y_i-\bar{y})^2
}.
$

A larger test-set value usually means that the model explains more variation. It does not prove that the model is causal or correct for every hour.

# Building the Demand-Prediction Model

We will compare:

1. a mean baseline,
2. linear regression,
3. random-forest regression.

The rows are hourly, but hours from the same calendar date are closely related. Every date must remain entirely inside one split.

We will use:

- 70% of unique dates for training,
- 15% for validation,
- 15% for testing.

Validation data selects the model. Test data provides the final evaluation.

In [ ]:
from sklearn.model_selection import (
    train_test_split,
)

from sklearn.compose import (
    ColumnTransformer,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
)

from sklearn.linear_model import (
    LinearRegression,
)

from sklearn.ensemble import (
    RandomForestRegressor,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.base import clone

print('Machine-learning tools imported.')

In [ ]:
CATEGORICAL_FEATURES = [
    'hour',
    'day_of_week',
    'season',
]

NUMERIC_FEATURES = [
    'is_holiday',
    'temperature_c',
    'humidity_percent',
    'wind_speed_m_s',
    'rainfall_mm',
    'snowfall_cm',
]

FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
)

TARGET = 'rented_bike_count'

print('Model inputs:')

for feature in FEATURES:
    print('-', feature)

print('\nPrediction target:')
print('-', TARGET)

## STUDENT TODO 1 — Split complete dates

Create training, validation, and test sets.

Requirements:

- split the unique dates rather than individual rows,
- use `random_state=RANDOM_SEED`,
- place 70% of dates in training,
- divide the remaining 30% equally between validation and testing,
- create `X_train`, `y_train`, `X_validation`, `y_validation`, `X_test`, and `y_test`.

In [ ]:
# STUDENT TODO 1: SPLIT COMPLETE DATES
# Suggested steps:
#   1. Get the unique dates.
#   2. Use train_test_split(..., test_size=0.30, random_state=RANDOM_SEED).
#   3. Split the remaining dates in half.
#   4. Select rows whose date belongs to each group.
#   5. Create the six X/y objects.

train_dates = None
validation_dates = None
test_dates = None

train_data = None
validation_data = None
test_data = None

X_train = None
y_train = None

X_validation = None
y_validation = None

X_test = None
y_test = None

print('Create the date-based training, validation, and test sets.')

In [ ]:
def check_date_split(
    train_dates,
    validation_dates,
    test_dates,
    train_data,
    validation_data,
    test_data,
    X_train,
    y_train,
    X_validation,
    y_validation,
    X_test,
    y_test,
):
    objects = [
        train_dates,
        validation_dates,
        test_dates,
        train_data,
        validation_data,
        test_data,
        X_train,
        y_train,
        X_validation,
        y_validation,
        X_test,
        y_test,
    ]

    if any(
        value is None
        for value in objects
    ):
        print('The date split is not complete.')
        return False

    train_date_set = set(
        pd.to_datetime(train_dates)
    )

    validation_date_set = set(
        pd.to_datetime(validation_dates)
    )

    test_date_set = set(
        pd.to_datetime(test_dates)
    )

    disjoint = (
        train_date_set.isdisjoint(
            validation_date_set
        )
        and train_date_set.isdisjoint(
            test_date_set
        )
        and validation_date_set.isdisjoint(
            test_date_set
        )
    )

    all_dates_used = (
        train_date_set
        | validation_date_set
        | test_date_set
    ) == set(
        pd.to_datetime(
            data['date'].unique()
        )
    )

    correct_columns = (
        list(X_train.columns) == FEATURES
        and list(X_validation.columns) == FEATURES
        and list(X_test.columns) == FEATURES
        and y_train.name == TARGET
        and y_validation.name == TARGET
        and y_test.name == TARGET
    )

    print(
        'Training dates:',
        len(train_date_set),
    )

    print(
        'Validation dates:',
        len(validation_date_set),
    )

    print(
        'Test dates:',
        len(test_date_set),
    )

    print(
        'Training rows:',
        len(train_data),
    )

    print(
        'Validation rows:',
        len(validation_data),
    )

    print(
        'Test rows:',
        len(test_data),
    )

    passed = (
        disjoint
        and all_dates_used
        and correct_columns
    )

    print(
        'Passed!'
        if passed
        else 'Check date overlap, row selection, and feature columns.'
    )

    return passed


split_ready = check_date_split(
    train_dates,
    validation_dates,
    test_dates,
    train_data,
    validation_data,
    test_data,
    X_train,
    y_train,
    X_validation,
    y_validation,
    X_test,
    y_test,
)

## Establish a baseline

A useful model should improve on a simple alternative.

The baseline predicts the average training demand for every validation and test hour.

In [ ]:
baseline_value = None
baseline_validation_predictions = None
BASELINE_VALIDATION_MAE = None

if split_ready:
    baseline_value = y_train.mean()

    baseline_validation_predictions = np.full(
        shape=len(y_validation),
        fill_value=baseline_value,
    )

    BASELINE_VALIDATION_MAE = mean_absolute_error(
        y_validation,
        baseline_validation_predictions,
    )

    print(
        'Baseline prediction:',
        f'{baseline_value:.2f} rentals',
    )

    print(
        'Baseline validation MAE:',
        f'{BASELINE_VALIDATION_MAE:.2f} rentals',
    )

else:
    print('Finish STUDENT TODO 1 before calculating the baseline.')

## Preparing categorical and numerical inputs

A model cannot directly interpret words such as `Monday` or `Summer`.

`OneHotEncoder` converts each category into numerical indicator columns.

We treat `hour` as a category because demand does not simply increase in a straight line from hour 0 through hour 23.

Numerical weather variables pass through unchanged.

## STUDENT TODO 2 — Build the model pipelines

Create one shared `ColumnTransformer`, then create a dictionary called `candidate_models` containing:

- `Linear regression`
- `Random forest`

Each dictionary value should be a scikit-learn `Pipeline` with steps named:

- `preprocessing`
- `regression`

Recommended random-forest settings are shown in the code hints.

In [ ]:
# STUDENT TODO 2: BUILD THE PREPROCESSOR AND TWO MODEL PIPELINES
# Useful objects:
#   ColumnTransformer
#   OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#   Pipeline
#   LinearRegression
#   RandomForestRegressor
#   clone
#
# Recommended random forest:
#   n_estimators=200
#   max_depth=18
#   min_samples_leaf=2
#   random_state=RANDOM_SEED
#   n_jobs=-1

preprocessor = None
candidate_models = None

print('Build the preprocessing and candidate-model pipelines.')

In [ ]:
def check_candidate_models(
    preprocessor,
    candidate_models,
):
    if (
        preprocessor is None
        or candidate_models is None
    ):
        print('The candidate models are not complete.')
        return False

    correct_preprocessor = isinstance(
        preprocessor,
        ColumnTransformer,
    )

    required_names = {
        'Linear regression',
        'Random forest',
    }

    correct_names = (
        set(candidate_models.keys())
        == required_names
    )

    correct_pipelines = (
        correct_names
        and all(
            isinstance(
                model,
                Pipeline,
            )
            for model in candidate_models.values()
        )
    )

    correct_steps = (
        correct_pipelines
        and all(
            'preprocessing' in model.named_steps
            and 'regression' in model.named_steps
            for model in candidate_models.values()
        )
    )

    passed = (
        correct_preprocessor
        and correct_names
        and correct_pipelines
        and correct_steps
    )

    print(
        'ColumnTransformer included:',
        correct_preprocessor,
    )

    print(
        'Required model names included:',
        correct_names,
    )

    print(
        'Both values are pipelines:',
        correct_pipelines,
    )

    print(
        'Required pipeline steps included:',
        correct_steps,
    )

    print(
        'Passed!'
        if passed
        else 'Revise the preprocessing and model pipelines.'
    )

    return passed


models_ready = check_candidate_models(
    preprocessor,
    candidate_models,
)

## STUDENT TODO 3 — Compare models using validation data

For each candidate model:

1. fit it on `X_train` and `y_train`,
2. predict `X_validation`,
3. clip negative predictions at zero,
4. calculate validation MAE, RMSE, and R-squared,
5. store the fitted model,
6. build `validation_summary`,
7. sort the summary by validation MAE from smallest to largest.

The model with the smallest validation MAE will be selected.

In [ ]:
# STUDENT TODO 3: TRAIN AND COMPARE THE CANDIDATE MODELS
# Create:
#   validation_results       -> list of dictionaries
#   fitted_models            -> dictionary
#   validation_summary       -> sorted DataFrame
#
# Each results dictionary should use these keys:
#   'model'
#   'validation_mae'
#   'validation_rmse'
#   'validation_r_squared'

validation_results = None
fitted_models = None
validation_summary = None

print('Fit and compare both candidate models.')

In [ ]:
def check_validation_results(
    validation_results,
    fitted_models,
    validation_summary,
):
    if (
        validation_results is None
        or fitted_models is None
        or validation_summary is None
    ):
        print('The validation comparison is not complete.')
        return False

    required_columns = {
        'model',
        'validation_mae',
        'validation_rmse',
        'validation_r_squared',
    }

    correct_models = (
        set(fitted_models.keys())
        == {
            'Linear regression',
            'Random forest',
        }
    )

    correct_columns = required_columns.issubset(
        validation_summary.columns
    )

    correct_rows = (
        len(validation_summary) == 2
    )

    sorted_by_mae = (
        correct_columns
        and validation_summary[
            'validation_mae'
        ].is_monotonic_increasing
    )

    finite_metrics = (
        correct_columns
        and np.isfinite(
            validation_summary[
                [
                    'validation_mae',
                    'validation_rmse',
                    'validation_r_squared',
                ]
            ].to_numpy()
        ).all()
    )

    passed = (
        correct_models
        and correct_columns
        and correct_rows
        and sorted_by_mae
        and finite_metrics
    )

    if correct_columns:
        display(
            validation_summary.round(3)
        )

    print(
        'Passed!'
        if passed
        else 'Check model fitting, metric calculations, and sorting.'
    )

    return passed


validation_ready = check_validation_results(
    validation_results,
    fitted_models,
    validation_summary,
)

In [ ]:
BEST_MODEL_NAME = None
evaluation_model = None

if validation_ready:
    BEST_MODEL_NAME = (
        validation_summary
        .iloc[0]['model']
    )

    evaluation_model = (
        fitted_models[
            BEST_MODEL_NAME
        ]
    )

    print(
        'Model selected using validation MAE:'
    )

    print(
        BEST_MODEL_NAME
    )

else:
    print('Finish STUDENT TODO 3 before selecting the model.')

## STUDENT TODO 4 — Evaluate the selected model on test data

Use the selected model to predict `X_test`.

Create:

- `test_predictions`
- `TEST_MAE`
- `TEST_RMSE`
- `TEST_R2`
- `BASELINE_TEST_MAE`
- `MAE_IMPROVEMENT_PERCENT`

Clip negative predictions at zero before calculating the metrics.

In [ ]:
# STUDENT TODO 4: TEST THE SELECTED MODEL
# Useful functions:
#   evaluation_model.predict(...)
#   np.clip(...)
#   mean_absolute_error(...)
#   mean_squared_error(...)
#   np.sqrt(...)
#   r2_score(...)

test_predictions = None

TEST_MAE = None
TEST_RMSE = None
TEST_R2 = None

BASELINE_TEST_MAE = None
MAE_IMPROVEMENT_PERCENT = None

print('Evaluate the selected model on the held-out test dates.')

In [ ]:
def check_test_evaluation(
    test_predictions,
    test_mae,
    test_rmse,
    test_r2,
    baseline_test_mae,
    improvement_percent,
):
    values = [
        test_mae,
        test_rmse,
        test_r2,
        baseline_test_mae,
        improvement_percent,
    ]

    if (
        test_predictions is None
        or any(
            value is None
            for value in values
        )
    ):
        print('The test evaluation is not complete.')
        return False

    predictions = np.asarray(
        test_predictions
    )

    correct_length = (
        len(predictions)
        == len(y_test)
    )

    finite_values = (
        np.isfinite(predictions).all()
        and all(
            np.isfinite(value)
            for value in values
        )
    )

    nonnegative_predictions = (
        predictions >= 0
    ).all()

    improvement_matches = np.isclose(
        improvement_percent,
        (
            (
                baseline_test_mae
                - test_mae
            )
            / baseline_test_mae
            * 100
        ),
    )

    passed = (
        correct_length
        and finite_values
        and nonnegative_predictions
        and improvement_matches
    )

    test_summary = pd.DataFrame(
        {
            'metric': [
                'Baseline test MAE',
                'Selected-model test MAE',
                'Selected-model test RMSE',
                'Selected-model test R-squared',
                'MAE improvement over baseline (%)',
            ],
            'value': [
                baseline_test_mae,
                test_mae,
                test_rmse,
                test_r2,
                improvement_percent,
            ],
        }
    )

    display(
        test_summary.round(3)
    )

    print(
        'Passed!'
        if passed
        else 'Check prediction length, clipping, and metric formulas.'
    )

    return passed


evaluation_ready = check_test_evaluation(
    test_predictions,
    TEST_MAE,
    TEST_RMSE,
    TEST_R2,
    BASELINE_TEST_MAE,
    MAE_IMPROVEMENT_PERCENT,
)

## Reading the evaluation metrics

- **MAE** is the average absolute difference between predicted and actual hourly rentals.
- **RMSE** gives especially large mistakes more influence.
- **R-squared** measures how much variation in the test observations is explained.
- **Improvement over baseline** shows whether the selected model is more useful than always predicting the training average.

A strong average score does not mean every individual hour is predicted accurately.

In [ ]:
evaluation_data = None

if evaluation_ready:
    evaluation_data = test_data[
        [
            'datetime',
            'hour',
            'day_of_week',
            'season',
            'temperature_c',
            'rainfall_mm',
            'rented_bike_count',
        ]
    ].copy()

    evaluation_data = evaluation_data.rename(
        columns={
            'rented_bike_count':
                'actual_demand',
        }
    )

    evaluation_data[
        'predicted_demand'
    ] = test_predictions

    evaluation_data['error'] = (
        evaluation_data['predicted_demand']
        - evaluation_data['actual_demand']
    )

    evaluation_data['absolute_error'] = (
        evaluation_data['error']
        .abs()
    )

    display(
        evaluation_data
        .head(10)
        .round(2)
    )

else:
    print('Finish the four required TODOs before inspecting errors.')

In [ ]:
if evaluation_data is None:
    print('Finish the evaluation first.')

else:
    plt.figure(
        figsize=(7, 7)
    )

    plt.scatter(
        evaluation_data[
            'actual_demand'
        ],
        evaluation_data[
            'predicted_demand'
        ],
        alpha=0.3,
    )

    maximum_value = max(
        evaluation_data[
            'actual_demand'
        ].max(),
        evaluation_data[
            'predicted_demand'
        ].max(),
    )

    plt.plot(
        [0, maximum_value],
        [0, maximum_value],
        linestyle='--',
    )

    plt.xlabel('Actual hourly rentals')
    plt.ylabel('Predicted hourly rentals')
    plt.title('Actual and Predicted Bike Demand')

    plt.tight_layout()
    plt.show()

In [ ]:
if evaluation_data is None:
    print('Finish the evaluation first.')

else:
    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        evaluation_data['error'],
        bins=30,
        edgecolor='black',
    )

    plt.axvline(
        0,
        linestyle='--',
    )

    plt.xlabel(
        'Prediction error: predicted minus actual'
    )

    plt.ylabel(
        'Number of test observations'
    )

    plt.title(
        'Distribution of Prediction Errors'
    )

    plt.tight_layout()
    plt.show()

In [ ]:
if evaluation_data is None:
    print('Finish the evaluation first.')

else:
    largest_errors = (
        evaluation_data
        .sort_values(
            'absolute_error',
            ascending=False,
        )
        .head(15)
    )

    display(
        largest_errors.round(2)
    )

## Investigating errors

Discuss:

1. Did the largest errors predict too high or too low?
2. Do several large errors occur at similar hours?
3. Are rainy, snowy, unusually hot, or unusually cold observations difficult?
4. What information might explain these mistakes but be missing from the data?

In [ ]:
def extract_model_explanation(
    fitted_pipeline,
):
    """Return linear coefficients or random-forest feature importances."""
    fitted_preprocessor = (
        fitted_pipeline
        .named_steps[
            'preprocessing'
        ]
    )

    fitted_regressor = (
        fitted_pipeline
        .named_steps[
            'regression'
        ]
    )

    feature_names = (
        fitted_preprocessor
        .get_feature_names_out()
    )

    if hasattr(
        fitted_regressor,
        'feature_importances_',
    ):
        explanation = pd.DataFrame(
            {
                'feature':
                    feature_names,
                'importance':
                    fitted_regressor
                    .feature_importances_,
            }
        )

        explanation[
            'absolute_effect'
        ] = explanation[
            'importance'
        ]

        explanation_type = (
            'Random-forest feature importance'
        )

    else:
        explanation = pd.DataFrame(
            {
                'feature':
                    feature_names,
                'weight':
                    fitted_regressor.coef_,
            }
        )

        explanation[
            'absolute_effect'
        ] = (
            explanation['weight']
            .abs()
        )

        explanation_type = (
            'Linear-regression coefficient'
        )

    explanation = (
        explanation
        .sort_values(
            'absolute_effect',
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return (
        explanation_type,
        explanation,
    )

In [ ]:
model_explanation = None

if evaluation_ready:
    (
        explanation_type,
        model_explanation,
    ) = extract_model_explanation(
        evaluation_model
    )

    print(
        explanation_type
    )

    display(
        model_explanation
        .head(20)
        .round(4)
    )

else:
    print('Finish the evaluation before interpreting the model.')

## Interpreting model explanations

For linear regression, the table contains coefficients:

- positive weights push a prediction upward,
- negative weights push a prediction downward,
- large absolute weights have stronger effects, although units matter.

For a random forest, the table contains feature importance:

- larger values indicate greater use in tree decisions,
- importance does not show whether the relationship is positive or negative.

Neither type of explanation proves causation.

# Train the Final Demand Model

The test set was used once for final evaluation.

After evaluation, fit a fresh copy of the selected model on all available historical rows. This final model will power the demonstration prediction.

In [ ]:
final_model = None

if evaluation_ready:
    final_model = clone(
        candidate_models[
            BEST_MODEL_NAME
        ]
    )

    final_model.fit(
        data[FEATURES],
        data[TARGET],
    )

    print(
        'Final model trained using '
        'all available observations.'
    )

    print(
        'Final model type:',
        BEST_MODEL_NAME,
    )

else:
    print('Finish evaluation before training the final model.')

# Predict One Future Hour

The final demonstration accepts one user-provided test point.

You will provide:

- date,
- hour,
- holiday status,
- temperature,
- humidity,
- wind speed,
- rainfall,
- snowfall.

The code derives the weekday and season from the date, creates the model input, and predicts the number of rental transactions during that hour.

Inputs outside the historical numerical ranges receive a warning.

In [ ]:
def season_from_month(
    month,
):
    """Convert a month number into the season labels used by the dataset."""
    if month in [
        12,
        1,
        2,
    ]:
        return 'Winter'

    if month in [
        3,
        4,
        5,
    ]:
        return 'Spring'

    if month in [
        6,
        7,
        8,
    ]:
        return 'Summer'

    return 'Autumn'


def validate_prediction_inputs(
    scenario_date,
    hour,
    is_holiday,
    temperature_c,
    humidity_percent,
    wind_speed_m_s,
    rainfall_mm,
    snowfall_cm,
):
    """Validate user values and derive calendar features."""
    try:
        scenario_date = pd.Timestamp(
            scenario_date
        )

    except Exception as error:
        raise ValueError(
            'scenario_date must be a valid date, '
            "such as '2026-07-15'."
        ) from error

    if not (
        isinstance(
            hour,
            (int, np.integer),
        )
        and 0 <= hour <= 23
    ):
        raise ValueError(
            'hour must be an integer from 0 through 23.'
        )

    if is_holiday not in [
        0,
        1,
        False,
        True,
    ]:
        raise ValueError(
            'is_holiday must be 0 or 1.'
        )

    numerical_inputs = {
        'temperature_c':
            temperature_c,
        'humidity_percent':
            humidity_percent,
        'wind_speed_m_s':
            wind_speed_m_s,
        'rainfall_mm':
            rainfall_mm,
        'snowfall_cm':
            snowfall_cm,
    }

    for (
        input_name,
        input_value,
    ) in numerical_inputs.items():
        if not isinstance(
            input_value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        ):
            raise TypeError(
                f'{input_name} must be numerical.'
            )

    if not (
        0
        <= humidity_percent
        <= 100
    ):
        raise ValueError(
            'humidity_percent must be between 0 and 100.'
        )

    for input_name in [
        'wind_speed_m_s',
        'rainfall_mm',
        'snowfall_cm',
    ]:
        if numerical_inputs[
            input_name
        ] < 0:
            raise ValueError(
                f'{input_name} cannot be negative.'
            )

    day_of_week = (
        scenario_date.day_name()
    )

    season = season_from_month(
        scenario_date.month
    )

    return {
        'scenario_date':
            scenario_date,
        'hour':
            int(hour),
        'day_of_week':
            day_of_week,
        'season':
            season,
        'is_holiday':
            int(is_holiday),
        'temperature_c':
            float(temperature_c),
        'humidity_percent':
            float(humidity_percent),
        'wind_speed_m_s':
            float(wind_speed_m_s),
        'rainfall_mm':
            float(rainfall_mm),
        'snowfall_cm':
            float(snowfall_cm),
    }


def find_input_warnings(
    scenario,
):
    """Warn when numerical inputs fall outside historical ranges."""
    warnings = []

    checked_features = [
        'temperature_c',
        'humidity_percent',
        'wind_speed_m_s',
        'rainfall_mm',
        'snowfall_cm',
    ]

    for feature in checked_features:
        historical_minimum = (
            data[feature].min()
        )

        historical_maximum = (
            data[feature].max()
        )

        scenario_value = (
            scenario[feature]
        )

        if (
            scenario_value
            < historical_minimum
            or scenario_value
            > historical_maximum
        ):
            warnings.append(
                (
                    f'{feature}={scenario_value} '
                    'is outside the historical range '
                    f'{historical_minimum} through '
                    f'{historical_maximum}.'
                )
            )

    return warnings

In [ ]:
def predict_bike_demand(
    scenario_date,
    hour,
    is_holiday,
    temperature_c,
    humidity_percent,
    wind_speed_m_s,
    rainfall_mm,
    snowfall_cm,
):
    """Predict hourly bike-rental demand for one scenario."""
    if final_model is None:
        raise RuntimeError(
            'Finish the four required TODOs and train the final model first.'
        )

    scenario = validate_prediction_inputs(
        scenario_date=scenario_date,
        hour=hour,
        is_holiday=is_holiday,
        temperature_c=temperature_c,
        humidity_percent=humidity_percent,
        wind_speed_m_s=wind_speed_m_s,
        rainfall_mm=rainfall_mm,
        snowfall_cm=snowfall_cm,
    )

    model_input = pd.DataFrame(
        [
            {
                feature:
                    scenario[feature]
                for feature in FEATURES
            }
        ]
    )

    prediction = float(
        final_model.predict(
            model_input
        )[0]
    )

    prediction = max(
        0,
        prediction,
    )

    input_warnings = find_input_warnings(
        scenario
    )

    print('=' * 70)
    print('BIKE-DEMAND PREDICTION')
    print('=' * 70)

    print(
        'Date:',
        scenario[
            'scenario_date'
        ].date(),
    )

    print(
        'Day:',
        scenario[
            'day_of_week'
        ],
    )

    print(
        'Hour:',
        f"{scenario['hour']:02d}:00",
    )

    print(
        'Season:',
        scenario[
            'season'
        ],
    )

    print(
        'Holiday:',
        'Yes'
        if scenario['is_holiday'] == 1
        else 'No',
    )

    print(
        'Temperature:',
        f"{scenario['temperature_c']:.1f} °C",
    )

    print(
        'Humidity:',
        f"{scenario['humidity_percent']:.1f}%",
    )

    print(
        'Wind speed:',
        f"{scenario['wind_speed_m_s']:.1f} m/s",
    )

    print(
        'Rainfall:',
        f"{scenario['rainfall_mm']:.1f} mm",
    )

    print(
        'Snowfall:',
        f"{scenario['snowfall_cm']:.1f} cm",
    )

    print('-' * 70)

    print(
        'Predicted hourly rentals:',
        round(prediction),
    )

    if TEST_MAE is not None:
        rough_low = max(
            0,
            prediction - TEST_MAE,
        )

        rough_high = (
            prediction + TEST_MAE
        )

        print(
            'Prediction plus/minus typical test error:',
            f'{round(rough_low)} to {round(rough_high)} rentals',
        )

        print(
            'Test MAE:',
            f'{TEST_MAE:.1f} rentals',
        )

    if input_warnings:
        print('\nWarnings:')

        for warning in input_warnings:
            print('-', warning)

    print(
        '\nThe error range is based on test MAE. '
        'It is not a formal confidence interval.'
    )

    return {
        'scenario': scenario,
        'model_input': model_input,
        'predicted_hourly_rentals': prediction,
        'input_warnings': input_warnings,
    }

## Demonstration cell

Edit the values below to create one test point, then run the cell.

A strong demonstration compares several sensible scenarios, such as:

- morning versus evening,
- dry versus rainy weather,
- weekday versus weekend,
- mild versus very cold temperatures.

In [ ]:
# USER-PROVIDED TEST DATAPOINT
# Change these values before running the demonstration.

prediction_result = predict_bike_demand(
    scenario_date='2026-07-15',
    hour=17,
    is_holiday=0,
    temperature_c=29.0,
    humidity_percent=72.0,
    wind_speed_m_s=2.1,
    rainfall_mm=0.0,
    snowfall_cm=0.0,
)

## Reflection

1. At which hours was average demand highest?
2. Why did complete dates remain inside one split?
3. What does one-hot encoding do?
4. Which model had the lowest validation MAE?
5. Why was the test set not used to select the model?
6. Did the selected model beat the baseline?
7. Explain the test MAE in the context of hourly rentals.
8. Which test observations produced the largest errors?
9. Which inputs appeared most important?
10. Why does importance or a coefficient not prove causation?
11. What happens when a demonstration input is outside the historical range?
12. Which important real-world feature is absent from this dataset?

## Optional controlled experiment

Change exactly one design choice and compare the results:

- remove one input feature,
- add `visibility_m`,
- add `solar_radiation_mj_m2`,
- change one random-forest setting,
- compare a different validation metric,
- change the date split,
- train separate weekday and weekend models,
- test several demonstration scenarios.

Record validation MAE, test MAE, RMSE, R-squared, and your interpretation.